In [ ]:
# Logging
import logging
logging.basicConfig(level=logging.INFO)

import warnings
warnings.filterwarnings(action='ignore')

# Typing
from typing import Any, Generator, Iterable, Sequence, Optional, Union

# Stdlib
from ast import literal_eval

# I/O
import csv
import json
from pathlib import Path

# Numeric
import pandas as pd
import numpy as np

# Cheminformatics
from rdkit import Chem

# Signac workflow control
import signac
from signac import Project
from signac.job import Job

from flow import FlowProject

# Init.py

## Formatting training data and breaking into jobs

In [ ]:
MONO_DATA_DIR = Path('monomer_data_formatted')
# N_TO_SAMPLE : Optional[int] = None
N_TO_SAMPLE : Optional[int] = 12
random : bool = True

mono_data_file_name = 'PolyID_master_data.csv'
mono_data_path = MONO_DATA_DIR / mono_data_file_name
assert(mono_data_path.exists())

READERS_BY_EXT = {
    '.xlsx' : pd.read_excel,
    '.csv'  : pd.read_csv,
}
df_reader_fn = READERS_BY_EXT[mono_data_path.suffix] # don't use get() here; WANT a KeyError if invalid
monomer_df = df_reader_fn(mono_data_path, index_col=0)
monomer_df.replace(np.nan, None, inplace=True) # convert NaN values to JSON-serializable NoneType

if N_TO_SAMPLE is not None:
    if random:
        monomer_df = monomer_df.sample(N_TO_SAMPLE)
    else:
        monomer_df = monomer_df.head(min(N_TO_SAMPLE, len(monomer_df)))
display(monomer_df)

## Generate jobs for each statepoint, injecting all other parameters not in the dataset

In [3]:
from typing import Any, Generator, Iterable, TypeAlias
StringMap : TypeAlias = dict[str, str]

import re
from itertools import product as cartesian_product


def cartesian_grid(param_options : dict[str, Iterable[Any]]) -> Generator[dict[str, Any], None, None]:
    '''
    Takes a dict keyed by parameter names whose values contain
    possible values for each respective parameter
    
    Exhaustively generates dicts (keyed by the same parameter names) containing every
    unique combination of those parameter values, with exactly one value for each key
    '''
    for param_point in cartesian_product(*param_options.values()):
        yield {
            param_name : param_value
                for param_name, param_value in zip(param_options.keys(), param_point)
        }

def parse_field_names_and_roles(dataframe : pd.DataFrame) -> tuple[StringMap, StringMap]:
    '''Extract the field (column) names and data role metadata
    Returns two dicts mapping from column names as-they-are to names and data roles, respectively'''

    HEADER_ROLE_RE = re.compile('(?P<field_name>.*?)<(?P<field_role>.*?)>') # role is delimited by chevrons

    colname_tag_free  : StringMap = {}
    colname_data_role : StringMap = {}
    for colname in dataframe.columns:
        matches = re.match(HEADER_ROLE_RE, colname)
        assert matches is not None

        match_fields : StringMap = matches.groupdict()
        colname_tag_free[colname] = match_fields['field_name']
        colname_data_role[colname] = match_fields['field_role']
        
    return colname_tag_free, colname_data_role

### Define non-dataset parameters

In [34]:
GRIDSPEC_PATH = Path('parameter_gridspec.json')
gridspec = { # define other parameters to sweep here!
    'DOP' : [
        3,
        # 5,
    ],
    'n_atoms_max' : [
        # 10_000,
        20_000,
    ],
    'pcharge_method' : [
        'Espaloma-AM1-BCC',
    ],
    'minimize_oligomer' : [
        True,
    ],
    'force_field' : [
        'openff_unconstrained-2.0.0.offxml',
        # 'openff-2.0.0.offxml',
    ],
    'use_switching_function' : [
        False,
        # True,
    ],
    'nonbonded_cutoff_nm' : [
        0.0,
    ],
    'box_padding_nm' : [
        0.9,
    ],
}
with GRIDSPEC_PATH.open('w') as file:
    json.dump(gridspec, file, indent=4)

In [ ]:

from collections import defaultdict

project_path = Path('polyid_test')
project = signac.init_project(project_path)

# populate data into statepoints
field_name, field_role = parse_field_names_and_roles(monomer_df)
for i, row in monomer_df.iterrows():
    # divvy up values according to field role
    rowdata = defaultdict(dict)
    for field, value in row.items():
        rowdata[field_role[field]][field_name[field]] = value

    # generate job statepoints and metadata, inject shared state parameters as needed
    for shared_params in cartesian_grid(gridspec):
        statepoint = {**rowdata['statedata'], **shared_params} # make copies to avoi mutation of common data
        metadata   = {**rowdata['metadata']} # make copies to avoid mutation of common data

        job = project.open_job(statepoint=statepoint)
        job.document = metadata

# Project.py

### Logging - this should live in its own file and eventually be absorbed into polymerist.gentuisl.logutils

In [5]:
from contextlib import contextmanager
from functools import partial


TIMESTAMP_LOG = '%Y-%m-%d %H:%M:%S' # timestamp format to use for logging
LOG_FORMATTER = logging.Formatter('%(asctime)s.%(msecs)03d [%(levelname)-8s:%(module)16s:line %(lineno)-4d] - %(message)s', datefmt=TIMESTAMP_LOG) 

def format_error_for_log(error : Exception) -> str:
    '''Converts a raised Exception to a loggable string'''
    return f'{type(error).__name__}: {error!s}'

def create_file_logger(
        logfile_path : Union[str, Path],
        logger_name : str,
        level : int=logging.INFO,
        mode : str='a',
        formatter : logging.Formatter=LOG_FORMATTER,
        suppress_console_logs : bool=True,
    ) -> tuple[logging.Logger, logging.FileHandler]:
    '''Create a unique logger (bound to a file) for a Signac job'''

    # Create "mouthpiece" proxy logger which will handle log input
    logger = logging.getLogger(logger_name)
    if suppress_console_logs:
        ... # TODO: figure out how to suppress console logs (contextlib.redirect_stdout doesn't work!)

    # Create new handler to target file idempotently, returning a prior equivalent handler if one already exists
    for file_handler in logger.handlers:
        if file_handler.baseFilename == logfile_path:
            file_handler.mode = mode # TOSELF: not sure if this is safe?
            break # stop looking once a handler
    else:
        file_handler = logging.FileHandler(logfile_path, mode=mode) # make new handler if prior handler is not found
        
    # Configure handler properties
    file_handler.setLevel(level)
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    return logger, file_handler

@contextmanager
def redirect_to_logfile(
        logfile_path : Union[str, Path],
        logger_name : str,
        level : int=logging.INFO,
        mode : str='a',
        formatter : logging.Formatter=LOG_FORMATTER,
        suppress_console_logs : bool=True,
        aux_loggers : Optional[Sequence[logging.Logger]]=None,
    ) -> Generator[logging.Logger, None, None]:
    '''Initializes a to-file logger for a job, and collects logs '''
    # Create a file handler
    proxy_logger, file_handler = create_file_logger(
        logfile_path=logfile_path,
        logger_name=logger_name,
        level=level,
        mode=mode,
        formatter=formatter,
        suppress_console_logs=suppress_console_logs,
    ) 

    # Temporarily bind handler to auxiliary loggers and suppress stdout
    if aux_loggers is None:
        aux_loggers = []

    orig_levels : dict[str, int] = {}
    for aux_logger in aux_loggers:
        orig_levels[aux_logger.name] = aux_logger.getEffectiveLevel() # record initial level for reversion at end
        if suppress_console_logs:
            ... # TODO: figure out how to suppress console logs (contextlib.redirect_stdout doesn't work!)
        aux_logger.addHandler(file_handler)

    # Execute 
    try:
        yield proxy_logger # supply mouthpiece for wrapped context to log to
    except Exception as e:
        proxy_logger.error(format_error_for_log(e))
    finally:
        for aux_logger in aux_loggers:
            aux_logger.setLevel(orig_levels[aux_logger.name])
            aux_logger.removeHandler(file_handler)

### Pre-defining blacklisted atoms and monomers

In [ ]:
from polymerist.polymers.monomers import specification
from polymerist.smileslib import substructures

# ATOMS, MONOMERS, AND REACTION MECHANISMS WHICH ARE, FOR ONE REASON OR ANOTHER, NOT ALLOWED
BLACKLISTED_ATOM_QUERIES = {
    'silicon' : Chem.MolFromSmarts('[Si]'),
    'sulfur'  : Chem.MolFromSmarts('[S]'),
    'metal'   : substructures.SPECIAL_QUERY_MOLS['metal'],
    # 'halogen' : substructures.SPECIAL_QUERY_MOLS['halogen'],
}

BLACKLISTED_MONOMER_SMILES = [ # monomers which are, for one reason or another, disallowed
    'CC(C)(C)c1cc(c(Oc2ccc(cc2)N(c3ccc(N)cc3)c4ccc(N)cc4)c(c1)C(C)(C)C)C(C)(C)C',  # the extraordinary number of symmetries of this amine ("4-N-(4-aminophenyl)-4-N-[4-(2,4,6-tritert-butylphenoxy)phenyl]benzene-1,4-diamine")... 
    'CC(C)(C)c1cc(Oc2ccc(-c3ccc(N)cc3)cc2C(F)(F)F)c(C(C)(C)C)cc1Oc1ccc(-c2ccc(N)cc2)cc1C(F)(F)F' # ...mean it takes impractically long to isomorphism match during the Topology partition step
] # TODO: might try setting limit of <1000 automorphisms for automatic check (since this is the default limit for substructure matches)
BLACKLISTED_MONOMER_QUERIES = {}
for smiles in BLACKLISTED_MONOMER_SMILES:
    exp_spi = specification.expanded_SMILES(smiles, assign_map_nums=False)
    banned_mol = Chem.MolFromSmiles(exp_spi, sanitize=False)
    display(banned_mol)
    BLACKLISTED_MONOMER_QUERIES[smiles] = banned_mol

BLACKLISTED_MECHANISMS = [
    'imide',
    'vinyl'
]

### Signac project proper

In [16]:
from string import ascii_uppercase

# OpenFF toolkits
from openff.toolkit import Molecule, Topology
from openff.toolkit.utils.exceptions import (
    MoleculeParseError,
    UnassignedChemistryInPDBError,
    IncorrectNumConformersWarning,
)

# Signac imports
logger_signac = logging.getLogger('signac')
logger_signac_flow = logging.getLogger('signac.flow')
SIGNAC_LOGGERS = [
    logger_signac,
    logger_signac_flow,
]

for logger in SIGNAC_LOGGERS:
    logger.setLevel(logging.INFO)

# Custom (polymerist) imports
import polymerist as ps
from polymerist.genutils.importutils import submodule_loggers
POLYMERIST_LOGGERS = [logger for logger in submodule_loggers(ps).values() if logger is not None]

from polymerist.genutils.textual.encoding import hash_as_alphanumeric
from polymerist.maths.lattices.integral import CubicIntegerLattice

from polymerist.polymers.monomers import MonomerGroup
from polymerist.polymers.building import build_linear_polymer, mbmol_to_openmm_pdb

from polymerist.mdtools.openfftools import topology
from polymerist.mdtools.openfftools.partition import partition
from polymerist.mdtools.openfftools.partialcharge.molchargers import MolCharger

from polymerist.rdutils import rdkdraw
rdkdraw.set_rdkdraw_size(300, 3/2)

from polymerist.rdutils.rdcoords.tiling import rdmol_effective_radius
from polymerist.rdutils.reactions.reactions import AnnotatedReaction, BadNumberReactants
from polymerist.rdutils.reactions.reactors import PolymerizationReactor

import polybuild_utils


# PROJECT-WIDE FILE NAMES
LOG_FILE_NAME = 'build_logs.log'
FRAG_FILE_NAME = 'fragments.json'
OLIGOMER_PDB_NAME = 'oligomer.pdb'
OLIGOMER_SDF_NAME = 'oligomer.sdf'
MELT_NEAT_SDF_NAME = 'melt_neat.sdf'
MELT_NEAT_INC_NAME = 'melt_neat_interchange.pkl'

def redirect_job_to_logfile(job : Job) -> logging.Logger:
    '''Thin wrapper around redirect_job_to_logfile() which is compatible with Signac jobs'''
    return redirect_to_logfile(
        logfile_path=job.fn(LOG_FILE_NAME),
        logger_name=job.id,
        level=job.project.doc.log_level,
        aux_loggers=POLYMERIST_LOGGERS
    )

# HELPER FUNCTIONS TO LOAD CHEMICAL STRUCTURES FROM STATEPOINT/ACCESORY DATA
def load_job_rdmol(job : Job) -> Chem.Mol:
    '''Helper method for loading an RDKit molecule from the SMILES in a job's statepoint'''
    reactant_mol = Chem.MolFromSmiles(job.sp.smiles_explicit, sanitize=False) # CRITICAL that sanitize=False to avoid stripping
    Chem.SanitizeMol(reactant_mol, sanitizeOps=specification.SANITIZE_AS_KEKULE) # single, unified mol containing individual reactant as disconnected components

    return reactant_mol

def load_job_rxn(job : Job) -> AnnotatedReaction:
    '''Helper method for loading an RDKit molecule from the SMILES in a job's statepoint'''
    rxn = AnnotatedReaction.from_smarts(
        job.sp.rxn_smarts.replace('#0', '*') # NOTE: this is a hack which should be sanitized in the previous rxn assembly step
    )
    rxn.Initialize()
    n_warn, n_err = rxn.Validate()
    # assert n_err == 0

    return rxn

def load_job_topology(job : Job, sdf_pathname : str) -> Optional[Molecule]:
    '''Read and return an OpenFF Topology from well-formed SDF file,
    returning None if encoding or other errors are encountered'''
    if not job.isfile(sdf_pathname):
        return None
    
    with redirect_job_to_logfile(job) as logger:
        sdf_path = Path(job.fn(sdf_pathname))
        if sdf_path.suffix != '.sdf':
            logger.error('Only SDF files are allowed for loading molecules')
            return None

        try:
            return topology.topology_from_sdf(sdf_path)
        except UnassignedChemistryInPDBError: # special cases for known common errors
            logger.error('OpenFF will not load molecule with ambiguous stereochemistry')
            return None
        except MoleculeParseError: # special cases for known common errors
            logger.error('Empty or malformed SDF file, could not read structural data')
            return None

load_job_melt_neat_topology = partial(load_job_topology, sdf_pathname=MELT_NEAT_SDF_NAME)
def load_job_oligomer_molecule(job : Job) -> Optional[Molecule]:
    '''Check whether a topology has atomic partial charges assigned to it'''
    oligomer_top = load_job_topology(job, OLIGOMER_SDF_NAME)
    if oligomer_top is None:
        return None
    return topology.get_largest_offmol(oligomer_top)


# DEFINING THE SIGNAC PROJECT CLASS PROPER 
class PolyIDBuild(FlowProject):
    pass

# 0) SIMPLE VALIDATION CHECKS ON REACTION DATA
@PolyIDBuild.label
def atoms_valid(job : Job) -> bool:
    '''Check no banned atom types are present'''
    # 2) check that none of the monomers are blacklisted
    reactant_mol = load_job_rdmol(job)
    return not any(
        substructures.matching_labels_from_substruct_dict(
            reactant_mol,
            BLACKLISTED_ATOM_QUERIES,
        )
    ) # if any illegal atoms are detected in the current monomer, return and exit

@PolyIDBuild.label
def monomers_allowed(job : Job) -> bool:
    '''Check no banned atom monomer fragments are present'''
    reactant_mol = load_job_rdmol(job)
    return not any(
        substructures.matching_labels_from_substruct_dict(
            reactant_mol,
            BLACKLISTED_MONOMER_QUERIES
        )
    ) # Exclude any monomers which are structurally disallowed

@PolyIDBuild.label
def mechanism_copied(job : Job) -> bool:
    return 'mechanism' in job.doc

@PolyIDBuild.label
def mechanism_allowed(job : Job) -> bool:
    '''
    Check that the rxn mechanism type is not explicitly blacklisted
    '''
    return job.doc.mechanism not in BLACKLISTED_MECHANISMS


# 1) TEST FOR RXN TEMPLATE COMPLIANCE AND ENUMERATE CHEMICAL FRAGMENTS
polymerize = PolyIDBuild.make_group(name='polymerize')

@PolyIDBuild.label
def reactant_order_evaluated(job : Job) -> bool:
    return 'reactant_ordering' in job.doc

@PolyIDBuild.label
def matches_rxn_template(job : Job) -> bool:
    return reactant_order_evaluated(job) and (job.doc.reactant_ordering is not None)

@polymerize
@PolyIDBuild.pre(atoms_valid)
@PolyIDBuild.pre(monomers_allowed)
@PolyIDBuild.pre(mechanism_copied)
@PolyIDBuild.pre(mechanism_allowed)
@PolyIDBuild.post(reactant_order_evaluated)
@PolyIDBuild.operation
def determine_reactant_order(job : Job) -> None:
    '''
    Check that SMILES monomers are compatible with the 
    reaction template for the mechanism they claim to follow
    '''
    # 1) check that monomers fit a reaction template
    reactant_mol = load_job_rdmol(job)
    reactants = Chem.GetMolFrags(reactant_mol, asMols=True)
    rxn = load_job_rxn(job)

    with redirect_job_to_logfile(job) as logger:
        try:
            reactant_ordering = rxn.valid_reactant_ordering(reactants, as_mols=False)
            job.doc.reactant_ordering = reactant_ordering # set EVEN if found ordering is None, to indicate this check has already been done
        except BadNumberReactants as bnr_error: # temporarily intercept this error to mark the job as having been checked for post-conditions
            job.doc.reactant_ordering = None
            raise bnr_error # re-raise to propagate this error up to the logger context
        
        if reactant_ordering is not None:
            logger.info(f'Identified valid reactant ordering: {reactant_ordering}')
        else:
            logger.error(f'No valid ordering of reactants could be solved for the chosen "{job.doc.mechanism}" rxn template')

ALLOWED_FUNCTIONALITIES : set[int] = {2}

@PolyIDBuild.label
def monomers_satisfy_functionality(job : Job) -> bool:
    '''
    Check that all monomers have allowed degrees of functionalization
    '''
    if not matches_rxn_template(job):
        return False
    
    rxn = load_job_rxn(job)
    reactant_smiles_all = job.sp.smiles_explicit.split('.')

    with redirect_job_to_logfile(job) as logger:
        for i in job.doc.reactant_ordering:
            reactant_smiles = reactant_smiles_all[i]
            reactant_mol = Chem.MolFromSmiles(reactant_smiles, sanitize=False) # CRITICAL that sanitize=False to avoid stripping
            Chem.SanitizeMol(reactant_mol, sanitizeOps=specification.SANITIZE_AS_KEKULE) # single, unified mol containing individual reactant as disconnected components
            
            num_funct_groups = substructures.num_substruct_queries_distinct(reactant_mol, rxn.GetReactantTemplate(i))
            if num_funct_groups not in ALLOWED_FUNCTIONALITIES:
                logger.error(f'Found {num_funct_groups} active functional groups (vs any from {ALLOWED_FUNCTIONALITIES}) for molecule {reactant_smiles}')
                return False
        else:
            return True
    
@polymerize
@PolyIDBuild.pre(matches_rxn_template)
@PolyIDBuild.pre(monomers_satisfy_functionality)
@PolyIDBuild.post.isfile(FRAG_FILE_NAME)
@PolyIDBuild.operation
def enum_fragments(job : Job) -> None:
    '''Enumerate all possible repeat unit fragment using cheminformatic reaction procedure'''
    rxn = load_job_rxn(job)
    reactor = PolymerizationReactor(rxn)

    reactant_mol = load_job_rdmol(job)
    reactants = Chem.GetMolFrags(reactant_mol, asMols=True)

    monogrp = MonomerGroup()
    with redirect_job_to_logfile(job) as logger:
        for intermediates, frags in reactor.propagate(reactants):
            for assoc_group_name, rdfragment in zip(ascii_uppercase, frags):
                # generate spec-compliant SMARTS
                raw_smiles = Chem.MolToSmiles(rdfragment)
                exp_smiles = specification.expanded_SMILES(raw_smiles)
                spec_smarts = specification.compliant_mol_SMARTS(exp_smiles)

                # record to monomer group
                affix = 'TERM' if MonomerGroup.is_terminal(rdfragment) else 'MID'
                monogrp.monomers[f'{assoc_group_name}_{affix}'] = [spec_smarts]

        monogrp.to_file(job.fn(FRAG_FILE_NAME))
        logger.info('Successfully enumerated and cached repeat unit fragments')


# 2) BUILD POLYMER STRUCTURE
oligomerize = PolyIDBuild.make_group(name='oligomerize')

@oligomerize
@PolyIDBuild.pre.copy_from(determine_reactant_order)
@PolyIDBuild.pre.copy_from(enum_fragments)
@PolyIDBuild.pre.isfile(FRAG_FILE_NAME)
@PolyIDBuild.post.isfile(OLIGOMER_PDB_NAME)
@PolyIDBuild.operation
def build_oligomer_pdb(job : Job) -> None:
    '''Generate coordinates and build oligomer PDB file using mBuild'''
    seq = 'BA' # hard-coded for now, plan to make more flexible in the future
    monogrp = MonomerGroup.from_file(job.fn(FRAG_FILE_NAME))
    with redirect_job_to_logfile(job) as logger:
        # check for identical parallel oligomer jobs
        parallel_struct_jobs = project.find_jobs({
            f'sp.{attr}' : getattr(job.sp, attr)
                for attr in ('DOP', 'smiles_explicit')
        })
        for parallel_job in parallel_struct_jobs:
            if (parallel_job.id != job.id) and (parallel_job.isfile(OLIGOMER_PDB_NAME)):
                logger.info(f'Job {parallel_job.id} already has my oligomer PDB!')
                break

        # generate coordinates with mBuild hook
        polymer = build_linear_polymer(
            monomers=monogrp,
            DOP=2*(1 + (job.sp.DOP - 1)/len(seq)), # formula to convert target DOP (considering an AB pair as a repeat unit) to effective DOP in builder
            sequence=seq,
            energy_minimize=True, # TODO: add master config option for energy minimization at project level
        )
        mbmol_to_openmm_pdb(job.fn(OLIGOMER_PDB_NAME), polymer)
        logger.info('Successfully generated PDB structure file')


@PolyIDBuild.label
def matches_m2p_smiles(job : Job) -> bool:
    '''Check whether the resulting polymer agrees with the SMILES output of M2P (if data is provided)'''
    ...

@PolyIDBuild.label
def no_ring_piercing(job : Job) -> bool:
    ... # TODO: implement post-minimization bond length check

@PolyIDBuild.label
def chemical_info_assigned(job : Job) -> bool:
    '''Check whether a topology has atomic partial charges assigned to it'''
    return job.isfile(OLIGOMER_SDF_NAME) and (load_job_oligomer_molecule(job) is not None)

@oligomerize
@PolyIDBuild.pre.copy_from(build_oligomer_pdb)
@PolyIDBuild.pre.isfile(OLIGOMER_PDB_NAME)
@PolyIDBuild.post.isfile(OLIGOMER_SDF_NAME)
@PolyIDBuild.operation
def assign_chem_info(job : Job) -> None:
    '''Assign chemical information to bare PDB graph and export completely-specified system to SDF file'''
    monogrp = MonomerGroup.from_file(job.fn(FRAG_FILE_NAME))
    with redirect_job_to_logfile(job) as logger:
        offtop = Topology.from_pdb(job.fn(OLIGOMER_PDB_NAME), _custom_substructures=monogrp.monomers)
        if not partition(offtop):
            logger.error(f'Failed to produce residue partition with fragments for job {job.id}')
            return None # exit before writing SDF; will cause post-condition to not be meet
        topology.topology_to_sdf(job.fn(OLIGOMER_SDF_NAME), offtop)
        logger.info('Successfully generated chemically-explicit SDF structure file')

@oligomerize
@PolyIDBuild.pre(chemical_info_assigned)
@PolyIDBuild.post.true('r_eff') # this ought to be fine, as these values should never be Falsy
@PolyIDBuild.post.true('n_atoms_oligomer') # this ought to be fine, as these values should never be Falsy
@PolyIDBuild.post.true('elem_counts_oligomer') # this ought to be fine, as these values should never be Falsy
@PolyIDBuild.operation
def summarize_oligomer(job : Job) -> None:
    '''Compute simple summarizing info about an oligomer which streamline lattice packing,
    namely the number of atoms, the distribution of elementsm and the effect (max) radius'''
    offmol = load_job_oligomer_molecule(job)
    job.doc.r_eff = rdmol_effective_radius(offmol.to_rdkit()) # TODO: this shouldn't require an RDKit conversion, just access tot he conformer
    job.doc.n_atoms_oligomer = offmol.n_atoms
    job.doc.elem_counts_oligomer = polybuild_utils.elem_counts(offmol)

@PolyIDBuild.label
def partial_charges_assigned(job : Job) -> bool:
    '''Check whether a topology has atomic partial charges assigned to it'''
    offmol = load_job_oligomer_molecule(job)
    if offmol is None:
        return False
    
    return offmol.partial_charges is not None
    
@oligomerize
@PolyIDBuild.pre(chemical_info_assigned)
@PolyIDBuild.post(partial_charges_assigned) # TODO: fill this in with something more substantive!!
@PolyIDBuild.operation
def assign_partial_charges(job : Job) -> None:
    '''Generate coordinates and build oligomer PDB file using mBuild'''
    offmol = load_job_oligomer_molecule(job)
    charger_type = MolCharger.subclass_registry.get(job.sp.pcharge_method, None)
    if charger_type is None:
        return
        
    with redirect_job_to_logfile(job) as logger:
        charger = charger_type()
        cmol = charger.charge_molecule(offmol)
        topology.topology_to_sdf(job.fn(OLIGOMER_SDF_NAME), cmol.to_topology())


# 3) PACK LATTICE
pack_lattice = PolyIDBuild.make_group(name='pack_lattice') 

@PolyIDBuild.label
def lattice_sites_determined(job : Job) -> bool:
    '''Check whether lattice sites have been assigned for '''
    return 'lattice_sites' in job.data

@pack_lattice
@PolyIDBuild.pre(chemical_info_assigned) # don't need charges, only valid cornformer to pick sites
@PolyIDBuild.post.true('n_oligomers')
@PolyIDBuild.post.true('lattice_shape')
@PolyIDBuild.post(lattice_sites_determined)
@PolyIDBuild.operation
def determine_lattice_sites(job : Job) -> None:
    '''Choose smallest accomodating cubic lattice, randomly subsample sites, and scale appropriately to oligomer size'''
    int_lattice : CubicIntegerLattice = polybuild_utils.generate_uniform_subpopulated_lattice(
        max_num_atoms=job.sp.n_atoms_max,
        num_atoms_in_mol=job.doc.n_atoms_oligomer
    )
    job.doc.n_oligomers = int_lattice.n_points
    job.doc.lattice_shape = int_lattice.counts_along_dims_as_str()
    
    transform = 2.0 * job.doc.r_eff * np.eye(3, dtype=float) # uniform scaling of lattice which guarantees points are one effective diameter apart
    job.data.lattice_sites = int_lattice.linear_transformation(transform, as_coords=False) # save lattice sites to numpy array on disc

@PolyIDBuild.label
def neat_melt_packed(job : Job) -> bool:
    '''Check is packing of the neat melt was successful'''
    return job.isfile(MELT_NEAT_SDF_NAME) and (load_job_melt_neat_topology(job) is not None)

@pack_lattice
@PolyIDBuild.pre(partial_charges_assigned)
@PolyIDBuild.pre(lattice_sites_determined)
@PolyIDBuild.post(neat_melt_packed)
@PolyIDBuild.operation
def pack_oligomers_onto_lattice(job : Job) -> None:
    '''Clone, randomly rotate, and move oligomer onto predetermined lattice sites'''
    offmol = load_job_oligomer_molecule(job)
    with job.data:
        lattice_sites = job.data.lattice_sites[:]

    with redirect_job_to_logfile(job) as logger:
        tiled_offtop = topology.topology_from_molecule_onto_lattice(
            offmol,
            lattice_points=lattice_sites,
            rotate_randomly=True,
            unique_mol_ids=True
        )
        topology.topology_to_sdf(job.fn(MELT_NEAT_SDF_NAME), tiled_offtop)

@pack_lattice
@PolyIDBuild.pre(neat_melt_packed)
@PolyIDBuild.post(lambda job : 'box_vectors' in job.data))
@PolyIDBuild.operation
def determine_periodic_box(job : Job) -> None:
    ...

@pack_lattice
@PolyIDBuild.pre(neat_melt_packed)
@PolyIDBuild.post.isfile(MELT_NEAT_INC_NAME)
@PolyIDBuild.operation
def neat_melt_to_interchange(job : Job) -> None:
    ...

# 4) PREPARE AND EXECUTE MD EXPORTS
md_export = PolyIDBuild.make_group(name='md_export') 

In [17]:
project_path = Path('polyid_test')
project = PolyIDBuild.get_project(project_path)
project.doc.log_level = logging.INFO

In [ ]:
project.print_status(detailed=True)

#### Run Jobs

In [ ]:
project.run(
    names=[
        # 'polymerize',
        # 'oligomerize',
        'pack_lattice',
    ]
)

In [ ]:
from polymerist.genutils.textual.prettyprint import dict_to_indented_str

job = project.open_job(id='facf6095639cec23b3c615e944d79f54')
print(dict_to_indented_str(job.doc))

In [ ]:
for job in project:
     if job.isfile(MELT_ONLY_SDF_NAME):
          print(job)

In [28]:
job = project.open_job(id='ea8a9cfe4d2489b765a2513cdde3da66')

In [29]:
offtop = topology.topology_from_sdf(job.fn(MELT_ONLY_SDF_NAME))

In [30]:
offtop.to_file('test.pdb', file_format='pdb')

## MD Engine file write

In [ ]:
from abc import ABC, abstractmethod
from openff.interchange import Interchange
from polymerist.genutils.decorators.classmod import register_subclasses


@register_subclasses(key_attr='ENGINE')
class MDEngineExporter(ABC):
    '''For simplifying the process of '''
    def __init_subclass__(cls, **kwargs) -> None:
        '''Enforce class-level definition of "Engine" name attr in subclasses'''
        super().__init_subclass__()
        if not hasattr(cls, 'ENGINE'):
            raise NotImplementedError('No class attr "ENGINE" set for subclass')
        
    def __init__(self, interchange : Interchange) -> None:
        super().__init__()
        self.interchange = interchange

    @property
    def inc(self) -> Interchange:
        '''Alias of "self.interchange" for convenience'''
        return self.interchange
    
    @abstractmethod
    def write_inputs(*args, **kwargs) -> None:
        pass

    
# Concrete classes
class LAMMPSMDExporter(MDEngineExporter):
    ENGINE = 'LAMMPS'

class OpenMMMDExporter(MDEngineExporter):
    ENGINE = 'OpenMM'